In [140]:
import sys
import subprocess
import importlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [141]:
NORMAL_LABEL = "Benign"

In [142]:
BASE_DIR = Path("__file__").resolve().parent
ROOT_DIR = BASE_DIR.parents[1]
ALGORITHM_DIR = ROOT_DIR / "model" / "algorithms"
OUTPUT_DIR = ROOT_DIR / "saved_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [143]:
MODEL_FILES = {
    "isolation_forest": "model_isolation_forest",
    "autoencoder": "model_autoencoder",
    "kmeans": "model_kmeans",
}

In [144]:
def import_module_or_notebook(name):
    py = ALGORITHM_DIR / f"{name}.py"
    nb = ALGORITHM_DIR / f"{name}.ipynb"

    if not py.exists() and not nb.exists():
        raise FileNotFoundError(f"{name}.py or {name}.ipynb not found")

    if nb.exists():
        subprocess.run(
            [sys.executable, "-m", "nbconvert", "--to", "script", "--output", name, str(nb)],
            cwd=ALGORITHM_DIR,
            check=True,
            capture_output=True,
            text=True,
        )

    if str(ALGORITHM_DIR) not in sys.path:
        sys.path.insert(0, str(ALGORITHM_DIR))

    importlib.invalidate_caches()
    return importlib.import_module(name) if name not in sys.modules else importlib.reload(sys.modules[name])

In [145]:
def get_models(names=None):
    if names is None:
        names = list(MODEL_FILES)
    if isinstance(names, str):
        names = [names]
    return {name: import_module_or_notebook(MODEL_FILES[name]) for name in names}


In [146]:
def get_training_data():
    X, y_int = make_classification(
        n_samples=12000,
        n_features=84,
        n_informative=40,
        n_classes=8,
        n_clusters_per_class=1,
        random_state=42,
    )

    classes = ["Benign", "DDoS", "DoS", "Recon", "Web-based", "BruteForce", "Spoofing", "Mirai"]
    y = np.array([classes[i] for i in y_int])
    X = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])

    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15, stratify=y_temp, random_state=42)
    return X_train, X_val, X_test, y_train, y_val, y_test

In [147]:
def to_df(X):
    if isinstance(X, pd.DataFrame):
        return X.copy()
    X = np.asarray(X)
    return pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])

In [148]:
def train_model(name):
    X_train, X_val, _, y_train, y_val, _ = get_training_data()
    model_mod = get_models(name)[name]

    if name == "isolation_forest":
        return model_mod.train(
            X_train, y_train, X_val, y_val,
            normal_label=NORMAL_LABEL,
            model_kwargs={
                "n_estimators": 200,
                "max_samples": 256,
                "contamination": "auto",
                "max_features": 1.0,
                "random_state": 42,
            },
            tune_threshold=True,
            save_path=str(OUTPUT_DIR / "isolation_forest.joblib"),
        )

    if name == "autoencoder":
        return model_mod.train(
            X_train, y_train, X_val, y_val,
            normal_label=NORMAL_LABEL,
            model_kwargs={"hidden_dims": (64, 32, 16), "latent_dim": 8, "random_state": 42},
            train_kwargs={"num_epochs": 15, "batch_size": 512, "lr": 0.003},
            tune_threshold=True,
            save_path=str(OUTPUT_DIR / "autoencoder.pt"),
        )

    if name == "kmeans":
        return model_mod.train(
            X_train, y_train, X_val, y_val,
            normal_label=NORMAL_LABEL,
            model_kwargs={"n_clusters": 2, "random_state": 42, "n_init": "auto"},
            save_path=str(OUTPUT_DIR / "kmeans.joblib"),
        )

    raise ValueError(f"Unknown model: {name}")

In [149]:
def train_all():
    return {name: train_model(name) for name in MODEL_FILES}

In [150]:
def load_saved_model(name):
    if name == "isolation_forest":
        return joblib.load(OUTPUT_DIR / "isolation_forest.joblib")

    if name == "kmeans":
        return joblib.load(OUTPUT_DIR / "kmeans.joblib")

    if name == "autoencoder":
        ckpt = torch.load(OUTPUT_DIR / "autoencoder.pt", map_location="cpu", weights_only=False)
        auto_mod = get_models("autoencoder")["autoencoder"]
        model = auto_mod.build_autoencoder(ckpt["input_dim"], **ckpt["model_kwargs"])
        model.load_state_dict(ckpt["state_dict"])
        model.eval()
        return {
            "model": model,
            "threshold": ckpt["threshold"],
            "normal_label": ckpt["normal_label"],
        }

    raise ValueError(f"Unknown model: {name}")

In [151]:
def predict_one(X, name):
    X = to_df(X)
    X_np = X.to_numpy()
    obj = load_saved_model(name)

    if name == "isolation_forest":
        score = obj["model"].decision_function(X_np)
        pred_bin = (score < obj["threshold"]).astype(int)

    elif name == "autoencoder":
        Xt = torch.tensor(X_np, dtype=torch.float32)
        with torch.no_grad():
            out = obj["model"](Xt)
            recon = out[0] if isinstance(out, (tuple, list)) else out
            score = ((recon - Xt) ** 2).mean(dim=1).cpu().numpy()
        pred_bin = (score > obj["threshold"]).astype(int)

    elif name == "kmeans":
        dist = obj["model"].transform(X_np)
        score = dist.min(axis=1)
        threshold = obj.get("threshold", np.percentile(score, 95))
        pred_bin = (score > threshold).astype(int)

    else:
        raise ValueError(f"Unknown model: {name}")

    pred_lbl = np.where(pred_bin == 0, NORMAL_LABEL, "Attack")

    return {
        "label": pred_lbl,
        "binary": pred_bin,
        "score": score,
    }

In [152]:
def predict_all(X):
    X = to_df(X)

    score_cols = []
    label_cols = []
    votes = []

    for name in MODEL_FILES:
        result = predict_one(X, name)
        votes.append(result["binary"])
        score_cols.append(np.asarray(result["score"]))
        label_cols.append(np.asarray(result["label"], dtype=object))

    votes = np.vstack(votes).T
    final_bin = (votes.sum(axis=1) >= 2).astype(int)
    final_label = np.where(final_bin == 0, NORMAL_LABEL, "Attack")
    final_score = votes.mean(axis=1)

    X_array = X.to_numpy()
    # scores_array = np.column_stack(score_cols + [final_score]) for if you want to show scores for each model
    scores_array = np.column_stack(final_score)
    #labels_array = np.column_stack(label_cols + [final_label]) for if you want to show labels for each model
    labels_array = np.column_stack(final_label)

    return X_array, scores_array, labels_array

In [153]:
def predict_phase2(X, model_name=None, as_numpy=True):
    X = to_df(X)

    if model_name:
        result = predict_one(X, model_name)
        X_array = X.to_numpy()
        scores_array = np.asarray(result["score"]).reshape(-1, 1)
        labels_array = np.asarray(result["label"], dtype=object).reshape(-1, 1)
        return X_array, scores_array, labels_array

    return predict_all(X)

In [161]:
#trained_models = train_all()

In [160]:
#X_train, X_val, X_test, y_train, y_val, _ = get_training_data()
#X_arr, score_arr, label_arr = predict_phase2(X_test)